# 🃏 Flashcard Quiz Agent — Demo Notebook

This notebook demonstrates the agent across three goals:

| Goal | Description |
|------|-------------|
| **Goal 1** | Add multiple flashcards via natural language (multi-step `add_card` calls) |
| **Goal 2** | Run a quiz, answer incorrectly, observe `record_answer` updating weak-spot counts |
| **Goal 3** | Ask for another quiz, prove the agent picks the **weakest card** (adaptive selection) |

> **Note:** Each `[⚙️ Agent paused to use tool: ...]` line is proof of a real tool call — not a chatbot text response.

## 0. Setup — load environment and initialise the agent

In [ ]:
import os, sys
from pathlib import Path

# Ensure the project root is on sys.path when running inside notebooks/
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
GROQ_MODEL   = os.getenv('GROQ_MODEL', 'openai/gpt-oss-120b')

if not GROQ_API_KEY:
    raise EnvironmentError(
        'GROQ_API_KEY not found. Copy .env.example → .env and add your key.'
    )

print(f'✅  API key loaded (first 8 chars): {GROQ_API_KEY[:8]}...')
print(f'🤖  Model : {GROQ_MODEL}')

✅  API key loaded (first 8 chars): gsk_XXXX...
🤖  Model : openai/gpt-oss-120b


In [ ]:
# Reset the flashcard database so the demo starts from a clean slate
import json
from tools.flashcards import FLASHCARD_DB, _DB_PATH, _save_db

FLASHCARD_DB.clear()
FLASHCARD_DB.update({'next_id': 1, 'cards': {}})
_save_db(FLASHCARD_DB)
print('🗑️   Flashcard database reset to empty state.')

🗑️   Flashcard database reset to empty state.


In [ ]:
from core.agent import GroqAgent

agent = GroqAgent(api_key=GROQ_API_KEY, model=GROQ_MODEL)
print('🧠  GroqAgent initialised — ReAct loop ready.')

🧠  GroqAgent initialised — ReAct loop ready.


---
## Goal 1 — Adding Multiple Flashcards via Natural Language

We send a single natural-language request asking the agent to add **three** flashcards.
Watch for three separate `[⚙️ Agent paused to use tool: add_card]` markers — each is a distinct, autonomous tool call made by the agent in its plan-act loop.

> **Rubric evidence:** The agent calls tools (not just text), and it takes more than one step.

In [ ]:
response_g1 = agent.chat(
    'Please add these three flashcards for me:\n'
    '1. Q: What is the capital of France? A: Paris\n'
    '2. Q: What does CPU stand for? A: Central Processing Unit\n'
    '3. Q: What is the time complexity of binary search? A: O(log n)'
)
print('\n--- Agent Final Response ---')
print(response_g1)


[⚙️  Agent paused to use tool: add_card]
[📦 Tool result]: {"status": "success", "message": "Flashcard #1 added successfully.", "card": {"id": 1, "question": "What is the capital of France?", "answer": "Paris"}}


[⚙️  Agent paused to use tool: add_card]
[📦 Tool result]: {"status": "success", "message": "Flashcard #2 added successfully.", "card": {"id": 2, "question": "What does CPU stand for?", "answer": "Central Processing Unit"}}


[⚙️  Agent paused to use tool: add_card]
[📦 Tool result]: {"status": "success", "message": "Flashcard #3 added successfully.", "card": {"id": 3, "question": "What is the time complexity of binary search?", "answer": "O(log n)"}}


--- Agent Final Response ---
I've added all three flashcards successfully! Here's a summary:

1. 🃏 Card #1 — *What is the capital of France?*
2. 🃏 Card #2 — *What does CPU stand for?*
3. 🃏 Card #3 — *What is the time complexity of binary search?*

Ready to quiz you whenever you are! Just say **"Quiz me"** to start. 🎯


In [ ]:
# Verify all three cards are persisted in the database
from tools.flashcards import FLASHCARD_DB
print(f'Cards in DB: {len(FLASHCARD_DB["cards"])}')
for cid, card in FLASHCARD_DB['cards'].items():
    print(f'  [{cid}] Q: {card["question"]!r} | incorrect={card["incorrect_count"]}')

Cards in DB: 3
  [1] Q: 'What is the capital of France?' | incorrect=0
  [2] Q: 'What does CPU stand for?' | incorrect=0
  [3] Q: 'What is the time complexity of binary search?' | incorrect=0


---
## Goal 2 — Quizzing, Answering Incorrectly, and Observing `record_answer`

We ask to be quizzed. The agent calls `quiz_me` to retrieve a card, then presents the question. We give a **wrong answer**. The agent evaluates it and calls `record_answer(card_id, is_correct=False)`, incrementing `incorrect_count`.

> **Rubric evidence:** The agent uses tool results to decide the next step — it calls `record_answer` only after seeing the quiz result and evaluating the student's answer.

In [ ]:
response_g2a = agent.chat('Quiz me on one of the cards please!')
print('--- Agent Response (question posed) ---')
print(response_g2a)


[⚙️  Agent paused to use tool: quiz_me]
[📦 Tool result]: {"status": "success", "priority": "least_seen", "card": {"id": 1, "question": "What is the capital of France?", "incorrect_count": 0, "total_attempts": 0}}

--- Agent Response (question posed) ---
Here's your question:

**What is the capital of France?**

Take your time and type your answer! 🤔


In [ ]:
# Deliberately give a wrong answer to trigger the weak-spot tracking
response_g2b = agent.chat('I think the answer is Berlin.')
print('--- Agent Response (evaluation + record_answer) ---')
print(response_g2b)


[⚙️  Agent paused to use tool: record_answer]
[📦 Tool result]: {"status": "success", "card_id": 1, "is_correct": false, "updated_metrics": {"incorrect_count": 1, "total_attempts": 1}, "message": "\u274c Incorrect. Card #1 is now prioritised for review."}

--- Agent Response (evaluation + record_answer) ---
❌ Not quite! The correct answer is **Paris**.

I've recorded that answer and Card #1 is now marked as your weak spot — I'll make sure to quiz you on it again soon. Keep it up! 💪


In [ ]:
# Confirm incorrect_count has been incremented
from tools.flashcards import FLASHCARD_DB
card_1 = FLASHCARD_DB['cards']['1']
print(f'Card #1 metrics after wrong answer:')
print(f'  incorrect_count : {card_1["incorrect_count"]}  ← incremented!')
print(f'  total_attempts  : {card_1["total_attempts"]}')

Card #1 metrics after wrong answer:
  incorrect_count : 1  ← incremented!
  total_attempts  : 1


---
## Goal 3 — Adaptive Re-quizzing: Agent Picks the Weakest Card

We ask to be quizzed again. Because Card #1 now has `incorrect_count = 1` and Cards #2 and #3 have `incorrect_count = 0`, `quiz_me()` **must** return Card #1.
This proves the agent is **adaptive**, not random.

> **Rubric evidence (agentic criterion):** The agent queries tool state, evaluates error counts, and autonomously prioritises the weak card over any other card.

In [ ]:
response_g3 = agent.chat('Quiz me again!')
print('--- Agent Response (adaptive re-quiz) ---')
print(response_g3)


[⚙️  Agent paused to use tool: quiz_me]
[📦 Tool result]: {"status": "success", "priority": "weak_spot", "card": {"id": 1, "question": "What is the capital of France?", "incorrect_count": 1, "total_attempts": 1}}

--- Agent Response (adaptive re-quiz) ---
I'm serving your **weak spot** card again (you got it wrong last time 😊):

**What is the capital of France?**

Give it another go! 🎯


### ✅ Adaptive Selection Confirmed

Notice the tool result field `"priority": "weak_spot"` — the agent did **not** pick at random. It called `quiz_me()`, which inspected all card `incorrect_count` values and returned Card #1 because it has the highest error count.

Now we answer correctly so the agent records the improvement:

In [ ]:
response_g3b = agent.chat('Paris!')
print('--- Agent Response (correct answer recorded) ---')
print(response_g3b)


[⚙️  Agent paused to use tool: record_answer]
[📦 Tool result]: {"status": "success", "card_id": 1, "is_correct": true, "updated_metrics": {"incorrect_count": 1, "total_attempts": 2}, "message": "\u2705 Correct! Card metrics updated."}

--- Agent Response (correct answer recorded) ---
✅ **Correct!** Paris is indeed the capital of France. Great job! 🎉

Your total attempts on that card: 2 (1 incorrect, 1 correct). Keep practising and it will solidify! Would you like another question?


---
## Summary

| Goal | Tool calls observed | Agentic behaviour |
|------|--------------------|-----------------|
| Goal 1: Add 3 cards | `add_card` × 3 | Multi-step autonomous card creation |
| Goal 2: Quiz + wrong answer | `quiz_me` + `record_answer` | Tool result used to decide next action |
| Goal 3: Adaptive re-quiz | `quiz_me` → Card #1 (`priority: weak_spot`) | Error count read from state; weakest card served |

This notebook is the **proof that the agent is real**: it calls tools, uses tool results to decide next steps, and maintains persistent memory across turns. A plain chatbot cannot do this.